## Tujuan

Tahap ini bertujuan untuk mempersiapkan data agar dapat digunakan dalam proses pemodelan. Proses preprocessing meliputi transformasi data kategorikal menjadi numerik, serta normalisasi atau standarisasi data numerik agar memiliki skala yang seragam. Data yang digunakan telah melalui tahap Data Cleaning dan Feature Engineering untuk memastikan kualitas dan relevansi fitur, serta telah melewati tahap Exploratory Data Analysis (EDA) untuk memahami pola, distribusi, dan hubungan antar variabel dalam dataset.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/clean_data.csv')

In [2]:
df.head().T

,0,1,2,3,4
age,14,19,17,15,15
daily_social_media_hours,7.9,1.9,1.3,7.4,4.7
sleep_hours,7.4,8.0,7.6,6.9,4.9
screen_time_before_sleep,2.9,2.9,0.5,1.6,3.0
academic_performance,3.01,3.22,3.92,3.48,2.37
physical_activity,1.5,0.8,0.0,0.8,1.4
social_interaction_level,low,high,high,medium,medium
stress_level,2,8,2,1,3
anxiety_level,2,1,4,7,5
addiction_level,1,10,2,9,2


Kita menetapkan `digital_wellbeing_flag` sebagai target. Kolom yang berpotensi menyebabkan kebocoran data (*data leakage*) seperti skor risiko akan kita hapus.

In [3]:
# 1. Encoding Target
target_map = {'Healthy': 0, 'Moderate': 1, 'At Risk': 2}
df['target'] = df['digital_wellbeing_flag'].map(target_map)

# 2. Encoding fitur ordinal yang terlewat
interaction_map = {'low': 0, 'medium': 1, 'high': 2}
df['social_interaction_level'] = df['social_interaction_level'].map(interaction_map)

# 3. Memisahkan Fitur (X) dan Target (y)
drop_cols = ['digital_wellbeing_flag', 'target', 'mental_health_risk_score', 'risk_category', 'sleep_quality']
X = df.drop(columns=drop_cols, errors='ignore')
y = df['target']

print(f"[INFO] Dimensi Fitur (X): {X.shape}")
print(f"[INFO] Dimensi Target (y): {y.shape}")

[INFO] Dimensi Fitur (X): (1200, 17)
[INFO] Dimensi Target (y): (1200,)


Menyamakan skala angka menggunakan `StandardScaler`.

In [4]:
# Inisialisasi StandardScaler
scaler = StandardScaler()

# Fit dan Transform data fitur (X) -> Di sini X_scaled dibuat!
X_scaled_array = scaler.fit_transform(X)

# Mengembalikan array numpy menjadi DataFrame Pandas
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

# Menyimpan objek Scaler ke dalam folder models
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')

print("[INFO] Feature Scaling berhasil.")
print("[INFO] Objek Scaler disimpan di '../models/scaler.pkl'")
X_scaled.head()

[INFO] Feature Scaling berhasil.
[INFO] Objek Scaler disimpan di '../models/scaler.pkl'


,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,total_screen_exposure,sleep_efficiency,high_social_media_usage,active_lifestyle,sleep_quality_encoded,gender_male
0,-0.954099,1.657833,0.659177,1.618830,0.034026,0.834276,-1.191094,-1.187367,-1.272335,-1.613389,-0.162845,2.079017,-0.691494,1.161179,0.915244,0.270052,0.975305
1,1.519796,-1.299649,1.075244,1.618830,0.398282,-0.368593,1.286051,0.880116,-1.622199,1.567444,-0.162845,-0.678910,-0.524431,-0.861194,-1.092604,1.609155,-1.025320
2,0.530238,-1.595397,0.797866,-1.731436,1.612470,-1.743301,1.286051,-1.187367,-0.572609,-1.259963,-0.162845,-2.057873,2.750013,-0.861194,-1.092604,0.270052,-1.025320
3,-0.459320,1.411376,0.312455,-0.195897,0.849266,-0.368593,0.047479,-1.531947,0.476980,1.214018,-0.162845,1.251639,0.129901,1.161179,-1.092604,0.270052,0.975305
4,-0.459320,0.080509,-1.074435,1.758424,-1.076088,0.662437,0.047479,-0.842786,-0.222746,-1.259963,-0.162845,0.654088,-1.421701,-0.861194,0.915244,-1.069050,-1.025320


Data fitur yang sudah di-scale digabung kembali dengan target, lalu disimpan.

In [5]:
# Menggabungkan kembali X_scaled dan y
df_preprocessed = X_scaled.copy()
df_preprocessed['target'] = y.values

# Menyimpan ke file CSV
os.makedirs('../data', exist_ok=True)
preprocessed_path = '../data/preprocessed_data.csv'
df_preprocessed.to_csv(preprocessed_path, index=False)

print(f"[INFO] Data siap modeling berhasil disimpan di: {preprocessed_path}")
print(f"[INFO] Dimensi akhir data preprocessed: {df_preprocessed.shape[0]} baris, {df_preprocessed.shape[1]} kolom")

[INFO] Data siap modeling berhasil disimpan di: ../data/preprocessed_data.csv
[INFO] Dimensi akhir data preprocessed: 1200 baris, 18 kolom


## Kesimpulan

Pada tahap preprocessing, variabel kategorikal telah dikonversi menjadi numerik melalui teknik encoding, serta variabel numerik telah dinormalisasi menggunakan standarisasi. Data hasil preprocessing telah siap digunakan dalam proses pemodelan, baik untuk segmentasi pelanggan maupun prediksi respons kampanye.